In [2]:
# Генерируем воспроизводимый синтетический датасет успеваемости и сохраняем его в Parquet.
import numpy as np
import pandas as pd

SEED = 42
SAMPLES_PER_CLASS = 500

rng = np.random.default_rng(SEED)

# Восемь признаков, характеризующих учебную деятельность студента.
feature_names = [
    "attendance",  # Посещаемость, %
    "homework",  # Выполнение домашних заданий, %
    "tests",  # Результаты контрольных, баллы из 100
    "labs",  # Результаты лабораторных, баллы из 100
    "activity",  # Активность на занятиях, баллы из 10
    "self_study",  # Самостоятельная подготовка, часы в неделю
    "assignments",  # Своевременная сдача работ, %
    "missed_classes",  # Пропущенные занятия за семестр
]

class_names = ["troechnik", "horoshist", "otlichnik"]

# Средние значения признаков для троечников, хорошистов и отличников.
# Меньшее число пропусков соответствует более высокой успеваемости.
class_centers = np.array(
    [
        [65, 55, 52, 55, 3.5, 3, 55, 18],
        [82, 80, 75, 78, 6.5, 7, 82, 9],
        [96, 96, 93, 94, 9.0, 13, 97, 2],
    ],
    dtype=np.float32,
)

# Стандартные отклонения позволяют немного варьировать характеристики студентов.
# Разнос центров выбран достаточно большим для разделимости классов.
feature_std = np.array([3, 4, 4, 4, 0.35, 0.8, 4, 1.5], dtype=np.float32)

# Создаём объекты каждого класса вокруг своего центра.
features_by_class = []
labels_by_class = []

for class_id, center in enumerate(class_centers):
    samples = rng.normal(
        loc=center,
        scale=feature_std,
        size=(SAMPLES_PER_CLASS, len(feature_names)),
    )

    # Ограничиваем признаки допустимыми диапазонами.
    samples = np.clip(
        samples, [0, 0, 0, 0, 0, 0, 0, 0], [100, 100, 100, 100, 10, 40, 100, 60]
    )

    features_by_class.append(samples.astype(np.float32))
    labels_by_class.extend([class_id] * SAMPLES_PER_CLASS)

X = np.vstack(features_by_class)
y = np.asarray(labels_by_class, dtype=np.int64)

# Перемешиваем студентов, чтобы классы не шли отдельными блоками.
permutation = rng.permutation(len(y))
X = X[permutation]
y = y[permutation]

# Преобразуем класс в три one-hot столбца.
y_one_hot = np.eye(len(class_names), dtype=np.int64)[y]

df = pd.DataFrame(X, columns=feature_names)

for class_id, class_name in enumerate(class_names):
    df[f"grade_{class_name}"] = y_one_hot[:, class_id]

# Сохраняем все 8 признаков и 3 столбца целевого класса.
output_path = "../datasets/student_performance.parquet"
df.to_parquet(output_path, index=False)

print(f"Датасет сохранён: {output_path}")
print(f"Размер: {df.shape[0]} объектов, {df.shape[1]} столбцов")
display(df.head())

Датасет сохранён: ../datasets/student_performance.parquet
Размер: 1500 объектов, 11 столбцов


,attendance,homework,tests,labs,activity,self_study,assignments,missed_classes,grade_troechnik,grade_horoshist,grade_otlichnik
0,84.358971,84.858231,72.557571,75.903267,6.417617,6.885142,77.967903,10.898900,0,1,0
1,98.150200,99.825562,92.924957,90.735123,9.094598,12.813385,96.035950,1.231400,0,0,1
2,68.599190,56.878910,52.272404,60.921814,3.540313,2.756527,51.648708,19.054539,1,0,0
3,82.031723,83.181419,81.585976,80.341942,6.781160,6.840117,79.168808,9.980979,0,1,0
4,65.119080,58.842937,44.188740,47.104664,3.595831,2.994307,55.680706,17.306808,1,0,0
